In [1]:
import pathlib

import yaml
import scipy.signal
import pandas as pd
import numpy as np

import jax.random as jr
from jaxtyping import Array, Float

import torch
from torch.utils.data import DataLoader

import altair as alt

import trapallada.data
import trapallada.model.signal
import trapallada.model.image
import trapallada.util
import trapallada.core
import trapallada.plot

# import ipywidgets as widgets
# from sklearn.manifold import TSNE

So that `impor`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


The directory containing the data

In [3]:
data_directory = pathlib.Path.cwd().parent / 'data'
assert data_directory.exists()
data_directory

PosixPath('/home/manu/Sync/git/trapallada/data')

The particular dataset (specified by a *base* file name)

In [4]:
# base_filename = '4_classes_modulated'
# base_filename = '20_600'
base_filename = '63_225'

signals_file = data_directory / f'{base_filename}_signals.pt'
assert signals_file.exists()

# metadata_file = data_directory / f'{base_filename}_labels.parquet'
metadata_file = data_directory / f'{base_filename}_metadata.parquet'
assert metadata_file.exists()

# signals_file, metadata_file

Dataset is loaded

In [5]:
# ds = trapallada.data.SignalsDataset(signals_file, metadata_file)
ds = trapallada.data.SignalsDataset(signals_file, metadata_file, normalize=True)
# len(ds)

trapallada.data - WARNING - dismissed 198 constant signals (that cannot be normalized); 14885 left


The directory containing the actual results

In [6]:
# results_directory = pathlib.Path.cwd()
results_directory = pathlib.Path.cwd() / 'results'
# results_directory = pathlib.Path('/tmp')
assert results_directory.exists()
results_directory

PosixPath('/home/manu/Sync/git/trapallada/experiments/results')

In [7]:
list(results_directory.glob('*.eqx'))

[PosixPath('/home/manu/Sync/git/trapallada/experiments/results/3.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4v3.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_20_4000_epochs.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/smaller.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/1.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/real_z_50_4000_epochs.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/5.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4v2.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/4.eqx'),
 PosixPath('/home/manu/Sync/git/trapallada/experiments/results/2.eqx')]

*Base* file name for the results (trained model)

In [8]:
# input_files_stem = pathlib.Path('/tmp') / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_200e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z10_500e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'z20_300e'
# input_files_stem = pathlib.Path.cwd().parent /'experiments' / 'new' / 'z_10_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_50_300_epochs' # <-------------------------------------
# input_files_stem = results_directory / 'new' / 'z_2_300_epochs' 
# input_files_stem = results_directory / 'new' / 'z_3_300_epochs' 
# input_files_stem = results_directory / 'new' / 'real_z_50_300_epochs'
# ---
# input_files_stem = results_directory / 'real_z_50_4000_epochs'
# input_files_stem = results_directory / 'real_z_20_4000_epochs'
# input_files_stem = results_directory / 'smaller'
# input_files_stem = results_directory / '1'
# input_files_stem = results_directory / '2' # <---- good
# input_files_stem = results_directory / '3'
# input_files_stem = results_directory / '4' # <---- good
input_files_stem = results_directory / '4v2' # <---- good
# input_files_stem = results_directory / '5' # <---- overfitting


In [9]:
# w_file_upload = widgets.FileUpload(
#     accept='.eqx',
#     multiple=False
# )
# display(w_file_upload)

In [10]:
# w_file_upload.value

From the above *stem*, several file names are derived

In [11]:
# parameters_file = pathlib.Path('parameters.yaml')
parameters_file = input_files_stem.with_suffix('.yaml')
assert parameters_file.exists()

# weights_file = pathlib.Path('foo.eqx')
weights_file = input_files_stem.with_suffix('.eqx')
assert weights_file.exists()

results_file = input_files_stem.with_suffix('.parquet')
assert results_file.exists()

In [12]:
with parameters_file.open() as f:
        parameters = yaml.safe_load(f)
# parameters
%pycat {parameters_file}

PRNG seeds:
  model initialization:
    decoder: 43
    encoder: 42
  sampling:
    training set: 14
    validation set: 15
  training dataloader: 43
  training-validation splitting: 42
encoder:
  input_size:
  - 1
  - 225
loss:
  x_sd: 0.25
model:
  channels:
  - 12
  - 12
  - 12
  dim_z: 50 
  kernel_sizes:
  - 64
  - 32
  - 16
  stride: 2
training:
  batch_size: 32
  dl_rng_key: 43
  learning_rate: 3.0e-05
  n_epochs: 20000
  train_test_split:
    test_size: 0.2


In [13]:
results_df = pd.read_parquet(results_file)
results_df

training                            validation                \
              loss loglikelihood         kl         loss loglikelihood   
0      1779.533023  -1761.267838  18.265183  1726.398649  -1708.381793   
1      1715.127553  -1697.502489  17.625066  1710.773578  -1693.449071   
2      1707.027618  -1690.194488  16.833127  1705.656494  -1689.213145   
3      1703.116683  -1687.297869  15.818812  1702.444763  -1687.175557   
4      1699.997328  -1685.545554  14.451773  1698.635984  -1684.982914   
...            ...           ...        ...          ...           ...   
14004  1140.611324  -1045.092710  95.518614  1186.310117  -1091.480477   
14005  1140.528665  -1044.902207  95.626458  1186.897791  -1091.605887   
14006  1140.500030  -1044.916379  95.583653  1185.406786  -1090.091849   
14007  1139.740557  -1044.169761  95.570793  1186.889419  -1091.521527   
14008  1140.529511  -1044.952056  95.577456  1185.721333  -1090.693141   

                  
              kl  
0      18.016845  
1      17.324509  
2      16.443349  
3      15.269205  
4      13.653070  
...          ...  
14004  94.829637  
14005  95.291907  
14006  95.314943  
14007  95.367895  
14008  95.028192  

[14009 rows x 6 columns]

In [14]:
# next(results_df.iterrows())[1].to_dict()
# # {k:v for k,v in next(results_df.iterrows())[1].to_dict()}
# {'_'.join(k):v for k,v in next(results_df.iterrows())[1].to_dict().items()}

# Training curves

In [15]:
def metric_chart(df: pd.DataFrame, metric: str, n_rows: int = 4000):

    sub_df = df.loc[:n_rows,[('training', metric), ('validation', metric)]]
    sub_df.columns = sub_df.columns.get_level_values(0)
    chart = alt.Chart(
        sub_df.reset_index()
    ).transform_fold(
        sub_df.columns.to_list(), ['dataset', metric]
    ).mark_line().encode(
        x='index:Q', y=f'{metric}:Q', color='dataset:N'
    ).properties(width=600, height=400)

    return chart

metric_chart(results_df, 'loss') | metric_chart(results_df, 'loglikelihood') | metric_chart(results_df, 'kl')

alt.HConcatChart(...)

In [16]:
model = trapallada.model.signal.load_vae(weights_file, parameters_file)

Number of parameters

In [17]:
sum(trapallada.util.get_layers_n_parameters(model).values())

29789

Data loaders for both the training and validation sets

In [18]:
# train_dl, validation_dl = trapallada.data.dataloaders_from_dataset(
#     ds,
#     parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
#     parameters['training']['batch_size'], parameters['PRNG seeds']['training dataloader'])
(train_dl, validation_dl), (train_ds, validation_ds) = trapallada.data.dataloaders_from_dataset(
    ds,
    splitting_parameters=parameters['training']['train_test_split'] | dict(random_state=parameters['PRNG seeds']['training-validation splitting']),
    batch_size=parameters['training']['batch_size'], batching_random_state=parameters['PRNG seeds']['training dataloader'], return_datasets=True)

A batch from the **validation** set

In [19]:
# iter_dl = iter(train_dl) # training
iter_dl = iter(validation_dl) # validation
next(iter_dl), next(iter_dl)
batch = next(iter_dl)
batch[1].shape

torch.Size([32, 1, 225])

Is every class present?

In [20]:
batch[2].unique()

tensor([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
        nan, nan, nan, nan, nan, nan, nan, nan], dtype=torch.float64)

An element from the above batch is encoded, decoded, and a prediction plotted

In [21]:
# i = 0
# i = 1
i = 31

# indexes from the above batch
i_s, *_ = batch

def plot_prediction(ds, i):
    
    x = ds[i][1].numpy()
    x_pred, z_mean, z_sd = model(x, jr.PRNGKey(42))
    display(ds.metadata_df.loc[i])
    df = pd.DataFrame({'prediction': x_pred[0], 'actual': x.ravel()})
    n_rows = 2000
    
    return trapallada.plot.df(df.iloc[:n_rows])

# plot_prediction(ds, 25)
plot_prediction(ds, i_s[i].item())

start            2019-11-22 00:14:14.532324
end              2019-11-22 05:57:38.052324
tar file                     0853980901.tar
source      P0853980901PNS001SRCTSR8002.FTZ
Name: 169, dtype: object

alt.Chart(...)

## Javi's paper

In [22]:
javi_df = ds.metadata_df[ds.metadata_df['source'] == 'P0605950101PNS003SRCTSR8001.FTZ']
javi_df.tail()

,start,end,tar file,source
5914,2009-06-03 10:45:23.497454,2009-06-03 16:28:47.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5918,2009-06-03 12:02:02.497454,2009-06-03 17:45:26.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5922,2009-06-03 13:18:41.497454,2009-06-03 19:02:05.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5926,2009-06-03 14:35:20.497454,2009-06-03 20:18:44.017454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ
5930,2009-06-03 15:19:47.917454,2009-06-03 21:03:11.437454,0605950101.tar,P0605950101PNS003SRCTSR8001.FTZ


In [23]:
(javi_df['end'] - javi_df['start']).tail()

5914   0 days 05:43:23.520000
5918   0 days 05:43:23.520000
5922   0 days 05:43:23.520000
5926   0 days 05:43:23.520000
5930   0 days 05:43:23.520000
dtype: timedelta64[ns]

In [24]:
n_signals_to_plot = 1
trapallada.plot.signals_array(ds[javi_df.tail(n_signals_to_plot).index][1].numpy().squeeze(axis=1))

alt.Chart(...)

In [25]:
i_javi = javi_df.tail(1).index.item()
plot_prediction(ds, i_javi)

start            2009-06-03 15:19:47.917454
end              2009-06-03 21:03:11.437454
tar file                     0605950101.tar
source      P0605950101PNS003SRCTSR8001.FTZ
Name: 5930, dtype: object

alt.Chart(...)

Was this signal seen during training?

In [26]:
i_javi in train_ds[:][0]
# i_javi in validation_ds[:][0]

True

# Latent space projection

In [27]:
df = trapallada.util.make_projections_df(ds, validation_dl, model.encoder)
# df = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
10938,2005-04-21 22:27:05.000873,2005-04-22 04:10:28.520873,0305480101.tar,P0305480101PNS003SRCTSR8003.FTZ,-1.547519,2.423693,0.314902,-0.697773,0.565400,-0.019411,...,-1.733031,0.697771,-0.099259,-0.294151,-1.818153,-1.574399,0.376418,0.538555,-1.761247,-7.087225
5219,2012-03-19 15:40:27.798108,2012-03-19 21:23:51.318108,0671850301.tar,P0671850301PNS001SRCTSR8002.FTZ,-0.209110,-0.354794,-0.932366,0.245940,-0.222627,1.447778,...,-0.927741,-0.481954,-0.319890,-0.072136,0.852667,-1.302673,-1.357303,0.767743,-7.057368,8.933110
3745,2018-01-14 10:31:39.038451,2018-01-14 16:15:02.558451,0803240501.tar,P0803240501PNS003SRCTSR8004.FTZ,0.057235,-1.484896,0.289853,0.065160,0.285228,-0.306420,...,0.653351,-0.660286,-0.182555,0.066407,-0.560998,0.189607,-2.022089,-1.699681,7.137626,7.798099
7639,2022-08-18 19:33:53.144691,2022-08-19 01:17:16.664691,0902760801.tar,P0902760801PNS003SRCTSR8004.FTZ,-1.194842,-0.866654,-0.056033,-0.989524,-0.562469,0.053078,...,-3.105303,0.137007,-1.617524,-1.912367,0.002117,0.776567,-0.010258,-0.741586,-4.318361,5.816468
2301,2004-01-13 08:18:56.579036,2004-01-13 14:02:20.099036,0201160401.tar,P0201160401PNS003SRCTSR8005.FTZ,0.696073,-1.198007,-1.033855,-0.548720,0.038852,-0.854703,...,-0.091659,-0.470002,-0.962557,-1.010378,-0.215592,1.506949,-0.175467,0.386587,-6.382613,-6.159876


Projections also for the training set

In [28]:
df_train = trapallada.util.make_projections_df(ds, train_dl, model.encoder)
df_train.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,z_43,z_44,z_45,z_46,z_47,z_48,z_49,z_50,tsne_1,tsne_2
98,2018-01-18 06:39:08.007996,2018-01-18 12:22:31.527996,0810600201.tar,P0810600201PNS003SRCTSR8003.FTZ,0.100263,1.030232,1.328253,0.362666,-1.447433,0.050651,...,0.777403,0.293923,-0.282764,-0.330316,-1.184644,0.137353,1.292072,-0.548366,7.327216,10.717394
14544,2009-06-20 13:09:18.777333,2009-06-20 18:52:42.297333,0604310201.tar,P0604310201PNS003SRCTSR8004.FTZ,-0.592269,0.593737,0.312408,0.015412,1.195492,-1.111738,...,-0.195842,0.984096,0.025555,0.371388,0.261419,-0.398154,0.258840,-0.536937,17.978621,-37.783798
6279,2009-04-01 06:09:16.616242,2009-04-01 11:52:40.136242,0554750401.tar,P0554750401PNS003SRCTSR8004.FTZ,0.743860,-0.143750,0.164235,0.338120,-0.914375,1.503688,...,0.541974,0.080069,0.939609,-0.169236,-0.206550,0.192411,-0.627817,-0.978064,-16.147421,-1.706765
3734,2018-01-14 08:52:00.338451,2018-01-14 14:35:23.858451,0803240501.tar,P0803240501PNS003SRCTSR8001.FTZ,-0.769246,2.226907,-0.120127,-1.248657,-1.425876,-0.534781,...,-0.257866,-0.585258,0.018563,0.374450,0.383983,1.096041,0.824585,0.097670,5.502291,11.065851
5164,2017-11-25 11:03:35.937376,2017-11-25 16:46:59.457376,0803952401.tar,P0803952401PNS003SRCTSR8004.FTZ,1.426336,-0.689012,-0.743223,0.252151,-0.771232,0.692497,...,-0.045989,0.707976,1.200762,0.662048,-0.722029,0.620987,0.429762,0.394272,10.111650,-9.737190


In [29]:
df_train.loc[i_javi, ['tsne_1', 'tsne_2']]

tsne_1     7.578943
tsne_2    45.253197
Name: 5930, dtype: object

In [30]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='tar file:N',
    fill='tar file:N',
    tooltip=['source:N']
    # tooltip=['tar file:N']
).properties(
    width=800, height=600
).interactive()

/home/manu/mambaforge/envs/trap/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)


alt.Chart(...)

Signals that are just noise have no SNR

In [31]:
df['SNR'].isna().sum()

KeyError: 'SNR'

As a *fix*, their SNR is set to the maximum

In [ ]:
df.loc[df['label'] == 'noise', 'SNR'] = df['SNR'].max()

...plotting

In [ ]:
alt.Chart(df.sample(n=1_000)).mark_point().encode(
    x='tsne_1:Q',
    y='tsne_2:Q',
    color='label:N',
    fill='label:N',
    size='SNR:Q',
    tooltip=['file name:N']
).properties(
    width=800, height=600
).interactive()

The latent space is projected into a 3D space

In [ ]:
df_3 = trapallada.util.make_projections_df(ds, validation_dl, model.encoder, n_projected_components=3)
df_3.head()

In [ ]:
import plotly.express as px
fig = px.scatter_3d(df_3, x='tsne_1', y='tsne_2', z='tsne_3', color='label', opacity=0.5, width=800, height=800)
fig.show()

# Real data

In [ ]:
# signals_file = data_directory / '0605950101.pt'
signals_file = data_directory / '0605950101_bin_size_3.pt'
assert signals_file.exists()

In [ ]:
real_ds = trapallada.data.SignalsDataset(signals_file, metadata_filename=None, normalize=True)

In [ ]:
real_ds[3:5]

Data loaders for both the training and validation sets

In [ ]:
torch_rng = torch.Generator().manual_seed(parameters['PRNG seeds']['training dataloader'])
real_dl = DataLoader(real_ds, parameters['training']['batch_size'], shuffle=True, generator=torch_rng)

A batch

In [ ]:
batch = next(iter(real_dl))
batch

In [ ]:
signals = batch[1]
signals.shape

Let us drop the last few measurements so that the signals have the appropriate length for the trained model

In [ ]:
signals = signals[..., :10_000]
signals.shape

In [ ]:
n_samples_to_plot = 2000
trapallada.plot.signals_array(signals.squeeze()[:, :n_samples_to_plot])

A signal to *upsample* signals

In [ ]:
def upsample_signals(signals: Float[Array, 'batch 1 time'], target_size: int) -> Float[Array, 'batch 1 time']:
    
    upsampling_factor = np.ceil(target_size / signals.shape[-1])
    
    return scipy.signal.resample_poly(signals, upsampling_factor, 1, axis=-1)[..., :target_size]

In [ ]:
# resampled_signals = upsample_signals(signals, 10_000)
# resampled_signals.shape

In [ ]:
# n_samples_to_plot = 2_000
# trapallada.plot.signals_array(resampled_signals[..., :n_samples_to_plot].squeeze())

In [ ]:
# resampled_signals[0].shape

In [ ]:
i = 12
x_pred, z_mean, z_sd = model(signals[i].numpy(), jr.PRNGKey(42))

# first dimension of `x_pred` is 1; below it's squeezed out
df = pd.DataFrame({'prediction': x_pred[0], 'actual': signals[i].ravel()})

trapallada.plot.df(df.iloc[:n_samples_to_plot])